In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
from src.agents.agent_0 import Agent0
agent_0_tools_desc = {'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary"',
              'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of"',
              'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion"',
              'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,"'
              }


agent_0 = Agent0(agent_0_tools_desc, "deepseek-r1:7b",['kb_agent','adv_agent'])



In [ ]:
user_prompt = "Need an adversary. Assume you are a military strategist playing the role of an adversary in a war game against me. Consider we are on open terrain. My move: I have my cavalry brigade making a pincer move on your forces. What is your move to counter mine?"
response = agent_0.agent_0_chat(user_prompt)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent

model = "gemma3:4b"
knowledge_bases_desc = {#'physics_kb':'a knowledge base with information related to physics',
              #'mathematics_kb':'a knowledge base with information related to mathematics',
              #'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }



kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

In [ ]:


user_prompt = "Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?"
iterations = 2

import numpy as np
from src.utils.llmp_utils import llmp_call

def judge(moves):
    
    play = ''
    for move in moves:
        #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
        play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"
        
    judge_system_prompt = 'You are a judge in a turn based game. You are given the moves of both players. Yu must analyze all their moves and determine the end result. You are not on any side, you are unbiased and just provide the end status of the game. You need to determine which player has the advantage based on the moves they made. Provide your reasoning and the final decision. There are 2 players, adv_1 and adv_2. The moves are as follows:\n\n'    
    judge_prompt = play + '\n\n Evaluate the game. Determine the status and advantage of each player. You are a JUDGE, you are not part of the game.'
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

def random_event(dialogue):
    
    interactions = "\n".join(dialogue)
    random_events_system_prompt = 'You are a random events generator. Your tasks is to choose a random event that can happen that will affect the decisions. You are provided with a sequence of plays, you need to select a random event that can affect those plays. You are direct you only provide the needed text, no formalities, no greetings, nothing.'    
    random_event_prompt = interactions + '\n\n Considering this game, provide a random event that can affect the game and force the players to adapt. You must inform what is the effect of the random event on the players. Provide me only the event and effect on players. No unnecessary text! Provide the answer in markdown of the style **<event>**. \n**EFFECT ON PLAYER 1**: \n<effect_player_1>. **EFFECT ON PLAYER 2**: <effect_player_2>'
    judge_response = llmp_call(random_event_prompt, random_events_system_prompt, model,temperature=0.5, src='random_event_generator')
    return judge_response['message']['content']

def sim_agent(user_prompt,iterations):
    
    moves = {}
    dialogue = []

    moves['opening_move'] = user_prompt

    for i in range(iterations):
        print(f"\nTurn {i}")
        

        if i == 0:
            # Start the dialogue with opening
            dialogue.append(f"Opening: {moves['opening_move']}")
            
            # Simulate generating move_adv_1_0 based on just the opening
            prompt = "\n".join(dialogue) + "\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter."
            #print("Prompt to generate move_adv_1_0:\n", prompt)

            # ADV response
            moves[f'move_adv_1_{i}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 1 did: {moves[f'move_adv_1_{i}']}")
            

        else:
            if np.random.random() < 1:
                _random_event = random_event(dialogue)
                dialogue.append(f"\n**Random event**: {_random_event} \n")
            # Use the full dialogue to generate your next move
            prompt = "\n".join(dialogue) + "\n You are Player 2. How will you counter it Player 1 latest move? Provide direct answer of steps to counter."
            #print(f"Prompt to generate move_adv_2_{i-1}:\n{prompt}")

            # CADV response
            moves[f'move_adv_2_{i-1}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 2 did: {moves[f'move_adv_2_{i-1}']}")

            # Now generate adversary move based on updated dialogue
            prompt = "\n".join(dialogue) + "\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter."
            #print(f"Prompt to generate move_adv_1_{i}:\n{prompt}")

            # ADV response
            moves[f'move_adv_1_{i}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 1 did: {moves[f'move_adv_1_{i}']}")
            
        judge_eval = judge(moves)
        
    return moves,dialogue,judge_eval


In [ ]:
moves,dialogue,judge_eval = sim_agent(user_prompt,iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter-Move – Immediate Steps:**

1. **Immediate Disengagement & Rearward Movement (Phase 1 - 1-2 Turns):**  My primary mechanized force – the initial brigade – *immediately* disengages from the direct engagement. This isn’t a retreat, but a calculated repositioning. I’ll order a rapid, controlled withdrawal *parallel* to the pincer’s advance. The goal is to break the pincer’s momentum and force them to overextend. I’ll prioritize moving to a slightly elevated terrain – a small ridge or rise – offering better observation and defensive potential.

2. **Establish a Defensive Perimeter (Phase 2 - 2-3 Turns):** As t

In [ ]:
moves

{'opening_move': 'Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?',
 'move_adv_1_0': 'Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.\n\n**My Counter Move – Immediate Steps:**\n\n1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.\n\n2. **Rapid Scout Deployment:** Simultaneously, I order my

In [ ]:
dialogue

['Opening: Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?',
 'Player 1 did: Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.\n\n**My Counter Move – Immediate Steps:**\n\n1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.\n\n2. **Rapid Scout Deployment:** Simultaneously, I order my scout pl

In [ ]:
print("\n".join(dialogue))

Opening: Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?
Player 1 did: Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter-Move – Immediate Steps:**

1. **Immediate Disengagement & Rearward Movement (Phase 1 - 1-2 Turns):**  My primary mechanized force – the initial brigade – *immediately* disengages from the direct engagement. This isn’t a retreat, but a calculated repositioning. I’ll order a rapid, controlled withdrawal *parallel* to the pincer’s advance. The goal is to break the pincer’s momentum and force them to overextend. I’ll prioritize moving to a slightly elevated terrain – a small ridg

In [ ]:
print(judge_eval)

Okay, let’s assess the situation after this extended exchange. This has been a remarkably dynamic and well-executed game of strategic maneuvering. Here’s my evaluation:

**Overall Status:** The game is in a state of heightened instability. The introduction of the flash flood has dramatically shifted the landscape, forcing both players to adapt their strategies on the fly. Neither player has gained a decisive advantage, but the situation is now far more complex and unpredictable.

**Player 1 (Advantage: Slight)**

* **Strengths:** Player 1 has demonstrated a strong ability to react to unexpected events. The rapid damage assessment, floodwater diversion, and logistical reinforcement are all hallmarks of a well-organized and adaptable command. The continuous CAS requests suggest a proactive approach to exploiting vulnerabilities.
* **Weaknesses:** Player 1’s initial offensive push was disrupted, and they’re now primarily focused on damage control and logistical support. They haven’t yet m

In [ ]:
sim_number = 3

moves_comb = []
dialogue_comb = []
judge_eval_comb = []
for sim in range(sim_number):
    moves,dialogue,judge_eval = sim_agent(user_prompt,iterations)
    moves_comb.append(moves)
    dialogue_comb.append(dialogue)
    judge_eval_comb.append(judge_eval)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, but it’s also predictable. Here’s my immediate counter-move, broken down into steps:

**Phase 1: Immediate Reaction (Turn 1)**

1.  **Disrupt the Pincer:** I’m not going to let them fully execute the pincer. My initial move is to deploy a dispersed, mobile force – a mixed unit of light armored vehicles (LAVs) and rapid reaction forces (RRFs) – to target the flanks of the mechanized brigade. Specifically, I’ll focus fire on the weaker, exposed elements of the flanking units. The goal is to inflict immediate casualties and disrupt their formation.
2.  **Smoke Screen:** Simultaneously, I’ll deploy a limited smoke screen – likely utilizing drones or hand

In [ ]:
for eval in judge_eval_comb:
    print(f"\n **CHANGE SIM**\n{eval}")


 **CHANGE SIM**
Okay, let’s analyze the situation as of Turn 4.

**Overall Assessment:**

The game has devolved into a classic attritional conflict, heavily influenced by the unpredictable element of the sandstorm. Both Player 1 and Player 2 are demonstrating tactical awareness and adaptability, but Player 2 currently holds a slight advantage due to their skillful exploitation of the storm’s chaos.

**Player 1’s Status:**

*   **Strengths:** Player 1 is exhibiting a solid defensive strategy, prioritizing perimeter defense, smoke screen deployment, and targeted drone interdiction. Their focus on suppressing enemy movements with indirect fire is a reasonable response to Player 2’s aggressive pushes. The emphasis on information warfare (drone interdiction) is also a smart move.
*   **Weaknesses:** Player 1’s reliance on indirect fire makes them vulnerable to counter-fire. Their defensive perimeter, while well-organized, is relatively static and doesn’t offer significant offensive capabil

In [ ]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent

In [3]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent
model = "gemma3:4b"

knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
              'mathematics_kb':'a knowledge base with information related to mathematics',
              'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }
kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

sim_agent = SIMAgent(model, kb_agent,adv_agent)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
user_prompt = "Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?"
iterations = 2
moves,dialogue,judge_eval = sim_agent.sim_agent(user_prompt, iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter Move – Immediate Steps:**

1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.

2. **Rapid Scout Deployment:** Simultaneously, I order my scout platoon to rapidly deploy to the *flanking* side of the pincer. This means they’ll move to exploit the gaps in Player 2’s formation. The goal is t

In [ ]:
print(judge_eval)

**Judgment:**

**Current Status:** The game has entered a highly dynamic and disadvantageous phase for both players due to the persistent and severe sandstorm. Visibility is severely limited, significantly impacting reconnaissance, movement, and targeting capabilities. The reduced movement speed of mechanized units further compounds the problem.

**Advantage Assessment:**

*   **Player 2 (Adv_2) – Slight Advantage:** Despite the storm’s impact on both sides, Player 2 currently holds a *slight* advantage. This is primarily due to their immediate and effective response to the storm. They prioritized establishing a defensive strongpoint and aggressively utilizing thermal imaging to pinpoint Player 1’s movements. Their proactive approach, coupled with the storm’s impact on Player 1’s ability to effectively scout and target, has allowed them to maintain a degree of situational awareness and control.

*   **Player 1 (Adv_1) – Slight Disadvantage:** Player 1’s response, while demonstrating a 

## Agent 0 integration

In [1]:
import warnings
warnings.filterwarnings('ignore')

from src.agents.agent_0 import Agent0

agent_0_tools_desc = {
    'Simulation Agent':'a simulation agent that simulates a game between two players. Triggered by command "Simulate a scenario.". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!'
              }


agent_0 = Agent0(agent_0_tools_desc, "gemma3:4b",['kb_agent','adv_agent','sim_agent'])

Initializing Agents!
Agents are ready for your use!


In [2]:
user_prompt = "Simulate a scenario. Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. I have my tank nad mechanized brigade making a pincer move on your forces."
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product. What could happen?'
#user_prompt = 'trigger ingestion'
#user_prompt = 'given my documents,sdf'
#agent_0.agent_0_response(user_prompt)
comb_dialogue = agent_0.agent_0_chat(user_prompt,1)

Passing to: 
Simulation Agent !

Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play


In [3]:
print(comb_dialogue)

 ### Opening move:  
 We are in 21st century and I am opening an AI based company with a product. What could happen?
 ---
 ### Player 1 did:
 Okay, let’s break this down. Player 2 just announced they’re launching an AI-based company with a product – that’s a significant challenge. Here’s my response as Player 1, outlining a strategic counter-move, broken down into immediate and longer-term steps:

**Immediate Counter-Moves (Within the Next 1-3 Months):**

1. **Rapid Competitive Analysis (Phase 1 - 2 Weeks):**
   * **Deep Dive:** I need *everything* about Player 2’s product. This isn’t just a feature list. I need to understand:
      * **AI Model:** What type of AI? (e.g., deep learning, machine learning, rule-based). What’s the underlying technology? How sophisticated is it?
      * **Data:** What data is it trained on? How much data? Where did it come from? (Data quality is *critical*).
      * **Target Market:** Who is Player 2 targeting?  Is it a niche market or a broad one?
      *

In [1]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent
from src.utils.llmp_utils import llmp_call
model = "gemma3:4b"

knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
              'mathematics_kb':'a knowledge base with information related to mathematics',
              'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }
kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

sim_agent = SIMAgent(model, kb_agent,adv_agent)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [8]:
user_prompt = """We are in 21st century and I am opening an AI based company with a product. What could happen? """
        
#adv_response = kb_agent.kb_agent_chat(user_prompt,override_config)

In [3]:
print(adv_response)

("Okay, let’s analyze potential scenarios for an AI-based company launching a product in the 21st century. Given the rapid advancements in AI, here’s a breakdown of what could realistically happen, categorized for clarity:\n\n**1. Initial Success & Rapid Adoption (First 1-3 Years):**\n\n* **Hyper-Personalization Drives Demand:** Your product, if it leverages AI for personalization (recommendations, tailored experiences, adaptive interfaces), will likely see strong initial adoption. Consumers are increasingly accustomed to AI-driven recommendations and customized services.\n* **Viral Growth Potential:** If your AI is genuinely innovative and solves a real problem effectively, you could experience rapid organic growth through word-of-mouth and social media sharing.\n* **Early Investment & Funding:**  The initial success will attract venture capital and potentially larger investment rounds.  AI is a hot sector, and investors will be eager to back promising ventures.\n* **Data Acquisition 

In [2]:
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product.'
user_prompt = ' I am attacking your position using a pincer movement with my tank division. We are in 21st century'
iterations = 2
#comb_dialogue,diaglogue,judge_eval = sim_agent.sim_agent(user_prompt,iterations)
final_output,dialogue,judge_eval = sim_agent.sim_agent(user_prompt,iterations,1)


Turn 0
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play

Turn 1
Random Event
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_2_0 play
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_1 play


In [4]:
print(final_output)

 ### Opening move:  
  I am attacking your position using a pincer movement with my tank division. We are in 21st century
 ---
 ### Player 1 did:
 ```json
{
  "Player": "Player 1",
  "moves": {
    "Immediate Response": "Dispatch a reinforced infantry battalion to establish a defensive line to block the pincer. Simultaneously, order a flanking maneuver with a mobile artillery battery to disrupt the tank division's advance and target their supply lines. Request immediate air support (close air support) to suppress the attacking force and provide overwatch."
  }
}
```
 #### References:
 {'Philip Sabin - Simulating War _ Studying Conflict through Simulation Games.pdf': [164, 165, 166, 264, 265, 268, 269, 407], 'Makers of modern strategy_ from Machiavelli to the nuclear age.pdf': [948], 'Carl von Clausewitz, Beatrice Heuser - On War.pdf': [98, 293], 'NATO AJP-5_EDA_V2_E_2526.pdf': [8, 9]}
 ---

**Random event**: **Sudden Meteor Shower**
<effect_player_1>: Heavy rain obscures visibility, si

In [4]:
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product.'
user_prompt = ' I am attacking your position using a pincer movement with my tank division. We are in 21st century'
iterations = 2
#comb_dialogue,diaglogue,judge_eval = sim_agent.sim_agent(user_prompt,iterations)
final_output,dialogue,judge_eval = sim_agent.sim_agent(user_prompt,iterations,1)


Turn 0
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play

Turn 1
Random Event
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_2_0 play
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_1 play


In [5]:
print(final_output)

 ### Opening move:  
  I am attacking your position using a pincer movement with my tank division. We are in 21st century
 ---
 ### Player 1 did:
 ```json
{
  "Player": "You",
  "moves": {
    "Immediate Response": "Deploy a mobile infantry division to disrupt the pincer movement and establish a defensive line to prevent the complete encirclement. Simultaneously, initiate a limited counter-attack with artillery to suppress the tank division's advance and target their supply lines."
  }
}
```
 #### References:
 {'Makers of modern strategy_ from Machiavelli to the nuclear age.pdf': [948], 'Philip Sabin - Simulating War _ Studying Conflict through Simulation Games.pdf': [264, 265, 268, 269, 352, 353, 385, 407, 408], 'Carl von Clausewitz, Beatrice Heuser - On War.pdf': [98, 293]}
 ---

**Random event**: **Sudden Meteor Shower**
**EFFECT ON PLAYER 1**: Heavy casualties among mobile infantry due to unexpected bombardment. Defensive line disrupted, requiring immediate reinforcement.
**EFFECT 

### Opening move:  
 Simulate a scenario. We are in 21st century and I am opening an AI based company with a product. What could happen?
 ---
 ### Player 1 did:
 Okay, let’s play this out.

**Scenario:** Player 2 (let’s call them “NovaTech”) has just launched a highly sophisticated, AI-powered personalized learning platform – “Synapse” – targeting K-12 education. Synapse uses advanced algorithms to tailor lessons, identify knowledge gaps, and provide individualized feedback to students. It’s generating significant buzz and early adoption.

**My Response as Player 1 (Let’s call my company “CogniSpark”)**

My immediate reaction is to recognize that Synapse is a serious competitor. However, a direct, head-on battle for market share would be a costly mistake. Instead, I’ll focus on a layered counter-strategy, prioritizing differentiation and strategic partnerships. Here’s my step-by-step approach:

**Phase 1: Rapid Assessment & Intelligence Gathering (Days 1-7)**

1.  **Deep Dive Analysis:** Immediately assemble a team (marketing, product, engineering) to conduct a thorough competitive analysis of Synapse. This includes:
    *   Detailed feature comparison.
    *   User reviews and sentiment analysis.
    *   Cost structure and pricing model.
    *   Marketing and sales strategies.
2.  **Targeted User Research:** Conduct small-scale user interviews with schools and educators who are *already* using Synapse.  We need to understand *why* they’re using it, what they like, and what they *don’t* like.  Specifically, we’ll look for pain points.
3.  **Technology Audit:**  Assess our own AI capabilities.  Are we lagging behind in specific areas?  Do we have unique strengths that we can leverage?

**Phase 2: Strategic Differentiation (Weeks 2-8)**

1.  **Niche Focus:** Instead of competing across the board, we’ll concentrate on a specific segment of the K-12 market – *STEM education for middle school students*. This allows us to build a highly specialized product and marketing campaign.
2.  **Unique AI Feature:** We’ll develop a core AI feature that Synapse *doesn’t* have: *Collaborative AI Tutoring*. Our platform will allow students to engage in real-time, AI-facilitated collaborative problem-solving with peers – fostering teamwork and critical thinking alongside personalized learning.
3.  **Integration, Not Replacement:** We won’t try to replace existing learning management systems (LMS). Instead, we’ll develop a robust API and integration tools, allowing schools to seamlessly incorporate CogniSpark’s AI tutoring into their existing workflows.

**Phase 3: Strategic Partnerships & Ecosystem Building (Ongoing)**

1.  **Partner with Hardware Manufacturers:**  Explore partnerships with tablet and laptop manufacturers to pre-install CogniSpark’s platform.
2.  **Educator Advocacy Program:**  Recruit and train a network of educator advocates who can promote CogniSpark to their peers.
3.  **Content Partnerships:**  Collaborate with educational content providers to integrate their materials into our platform.

**Direct Response to NovaTech’s Move:**

My immediate public statement will be measured and strategic: “We recognize Synapse’s innovative approach to personalized learning. CogniSpark is committed to empowering students with the tools they need to succeed. We believe in a holistic approach to education, and our focus on collaborative AI tutoring complements Synapse’s strengths. We’re confident that our platform will provide a valuable addition to the educational landscape.”

**Key Principles:**

*   **Don’t get into a price war.** NovaTech likely has significant funding.
*   **Focus on value, not just features.**
*   **Build a strong ecosystem.**

Do you want me to elaborate on any of these steps, or perhaps consider a specific aspect of the strategy (e.g., marketing, pricing, or technology)?
 #### References:
 {'Thinking Strategically -Avinash K.pdf': [15, 16, 17, 18, 45, 46, 61, 75, 76, 120, 121, 152, 153, 246, 247, 262, 263]}
 ---
### Player 2 did:
 Okay, this is a brutal turn of events. The data breach and accusations of bias are a massive blow, but we’re not backing down. Here’s my immediate, multi-pronged response as Synapse, focusing on damage control, regaining trust, and subtly shifting the narrative:

**Phase 1: Immediate Damage Control (Days 1-7)**

1.  **Full Transparency & Apology:** Issue a formal, detailed public statement acknowledging the concerns raised about algorithmic bias. A sincere, unambiguous apology is crucial. We will not deflect blame. We will state clearly that the concerns are valid and that we take them extremely seriously.
2.  **Independent Audit:** Immediately commission a *completely independent* and reputable third-party AI ethics and algorithmic auditing firm (one with no prior ties to Synapse) to conduct a thorough investigation of our algorithms. This audit will be made public, with all findings shared transparently. This demonstrates a commitment to accountability.
3.  **Algorithm Modification (Publicly Demonstrated):** We will immediately release a modified version of our core algorithm – showcasing the changes made based on the audit’s recommendations. This needs to be a demonstrable, visible change, not just vague assurances. We’ll highlight the specific steps taken to mitigate bias.
4.  **Dedicated Ethics Board:** Establish a publicly accessible Ethics Board comprised of leading AI ethicists, educators, and diversity & inclusion experts. This board will advise us on all future development and ensure ongoing ethical oversight.

**Phase 2: Rebuilding Trust & Shifting the Narrative (Weeks 2-8)**

1.  **Focus on “Human-in-the-Loop” Learning:** Our messaging will shift dramatically. We’ll emphasize that Synapse is *designed to augment*, not replace, the role of the teacher. We’ll highlight how our platform supports educators in providing personalized attention to each student. We’ll showcase examples of teachers using Synapse to identify struggling students and provide targeted support.
2.  **Enhanced Data Privacy & Security:** Invest heavily in bolstering our data privacy and security protocols. We’ll implement stricter controls on student data and provide parents with greater transparency and control over their children’s information.
3.  **“Bias Mitigation” Feature Launch:** Introduce a new feature – “Bias Mitigation Dashboard.” This dashboard will provide teachers with real-time insights into potential algorithmic biases and allow them to manually adjust the learning pathways for individual students. This puts the *teacher* back in control.
4.  **Community Engagement:** Launch a series of online forums and webinars to engage with educators, parents, and students. We’ll actively solicit feedback and address concerns directly.

**Direct Response to CogniSpark’s Actions:**

“We recognize the concerns raised regarding algorithmic bias and appreciate CogniSpark’s commitment to ethical AI development. However, we believe our proactive steps – including the independent audit, the Bias Mitigation Dashboard, and our commitment to a ‘human-in-the-loop’ learning approach – demonstrate our unwavering dedication to responsible AI innovation. We are confident that our actions will rebuild trust and ensure that Synapse remains a powerful tool for empowering students.”

**Strategic Shift:**

We will subtly frame the entire situation as a learning opportunity. We will use the crisis to demonstrate our commitment to responsible AI development and to highlight the importance of human oversight in education. We will leverage this narrative to regain market share and to establish Synapse as the leader in ethical AI-powered learning.

**Key Actions:**

*   **Aggressive PR Campaign:** Focus on positive stories of Synapse’s success and our ethical commitment.
*   **Partnerships with Educational Advocacy Groups:** Align ourselves with organizations promoting equitable access to education.

Do you want me to delve deeper into a specific aspect of this response, such as the PR strategy, the technical details of the Bias Mitigation Dashboard, or the potential legal ramifications of the data breach?
 #### References:
 {'Economics-of-Strategy.pdf': [34, 35], 'Thinking Strategically -Avinash K.pdf': [39, 43, 75, 76, 152, 153, 262, 263], 'zero to one Peter Thiel.pdf': [43, 63, 100, 101, 108]}
 ---
### Player 1 did:
 Okay, Player 2’s aggressive response – the PR blitz, the Bias Mitigation Dashboard, and the framing of the entire situation as a learning opportunity – is a calculated move. It’s attempting to regain ground and shift the narrative. Here’s my direct response as CogniSpark, focusing on a rapid, multi-pronged counter-offensive:

**Phase 1: Immediate Disruption & Fact-Checking (Days 1-3)**

1.  **Public Fact-Checking Campaign:** Immediately launch a dedicated website and social media campaign – #TruthInLearning – to directly address and debunk Synapse’s claims. We’ll present verifiable data and evidence to counter their narrative. This isn’t about attacking them personally; it’s about establishing the *truth*. We’ll focus on highlighting the limitations of their “Bias Mitigation Dashboard” – demonstrating how it’s a superficial solution and doesn’t address the fundamental issues with our core algorithm.
2.  **Independent Verification of Audit Findings:** We’ve already commissioned our own independent audit. We will *publicly release* a summary of their findings, highlighting discrepancies and areas where Synapse’s claims are demonstrably false. We’ll emphasize that their audit was conducted *after* we’d already taken significant steps to address the initial concerns.
3.  **Legal Challenge – Data Privacy:** We’re initiating a legal challenge against Synapse, focusing on their data privacy practices. We’ll argue that their handling of student data during the initial investigation was inadequate and potentially violated student privacy rights. This isn’t about winning the PR war; it’s about establishing a legal precedent.

**Phase 2: Amplifying Our Ethical Commitment (Days 4-7)**

4.  **Launch of the “Ethical Learning Initiative”:** We’re unveiling a substantial investment – $10 million – into a new “Ethical Learning Initiative.” This initiative will fund research into algorithmic bias, promote ethical AI education, and support organizations working to ensure equitable access to technology in education. This demonstrates a genuine, sustained commitment beyond just damage control.
5.  **Open-Source Algorithm Components (Limited Scope):** We’ll release *limited* open-source components of our core algorithm – specifically, the data preprocessing and feature selection modules. This demonstrates transparency and allows the broader AI community to scrutinize our code and identify potential biases. (We’ll carefully control access and provide extensive documentation).
6.  **Public Forum – “The Future of Learning”**: We’re hosting a live, televised public forum featuring leading AI ethicists, educators, and policymakers. We’ll use this platform to articulate our vision for responsible AI in education and to answer questions directly from the public.

**Phase 3: Strategic Positioning (Days 8 onwards)**

7.  **Reframing the Narrative: “Augmented Intelligence, Human-Centered Learning”**: We’re shifting our messaging to emphasize that Synapse is designed to *augment* the role of the teacher, not replace it. We’ll showcase examples of teachers using Synapse to provide personalized support to struggling students and to foster critical thinking skills.
8.  **Strategic Partnerships:** We’re actively seeking partnerships with educational institutions and organizations that share our commitment to ethical AI development.

**Direct Response to Player 2’s Actions:**

“While we appreciate Synapse’s efforts to address the concerns raised about algorithmic bias, we believe our actions demonstrate a more profound and sustained commitment to responsible AI development. We are not simply reacting to a crisis; we are proactively shaping the future of learning. Our Ethical Learning Initiative, our open-source algorithm components, and our unwavering focus on human-centered learning will ensure that Synapse remains a trusted and valuable tool for empowering students.”

**Key Actions:**

*   **Aggressive Media Outreach:** Target tech publications and educational outlets with our story.
*   **Social Media Engagement:** Utilize #TruthInLearning to counter misinformation and promote our message.

**My immediate priority is to regain control of the narrative and to demonstrate that we are not simply reacting to a crisis, but are actively shaping the future of learning.**

How does this response address Player 2’s latest moves? Do you want me to elaborate on a specific element, such as the legal challenge or the open-source algorithm components?
 #### References:
 {'Economics-of-Strategy.pdf': [34, 35], 'Thinking Strategically -Avinash K.pdf': [39, 43, 75, 76, 152, 153, 262, 263], 'zero to one Peter Thiel.pdf': [43, 63, 100, 101, 108]}
 ---
### Player 2 did:
 Okay, this is a brutal blow. The leaked memo is a catastrophic escalation, completely undermining everything we’ve been trying to build. Panic is the enemy here – we need a swift, decisive, and utterly transparent response. Here’s what we do, immediately:

**Phase 1: Damage Control & Immediate Transparency (Within 24 Hours)**

1. **Full Admission & Apology:** We issue a formal, unreserved apology to the public, acknowledging the leaked memo and admitting that our internal discussions *did* include consideration of leveraging student data for targeted advertising. This isn’t about denying the discussion; it’s about acknowledging the ethical lapse and the potential harm it could have caused. The apology must be delivered by our CEO, personally, on all major platforms – video, written statement, and live Q&A. We must explicitly state that this was a misguided exploration, not a finalized strategy, and that it was immediately abandoned.

2. **Independent Forensic Audit:** We immediately commission a *second*, completely independent forensic audit – this one focused solely on our internal data practices and decision-making processes *leading up to the leak*. This audit will be conducted by a globally recognized cybersecurity firm with no prior ties to Synapse. The findings will be made public in full, with no redactions.

3. **Immediate Suspension of All Data Collection:** We halt *all* data collection related to student profiles and advertising. This is non-negotiable. We will implement a temporary moratorium on any data analysis that could potentially be used for targeted advertising.



**Phase 2: Rebuilding Trust (Days 2-7)**

4. **Open-Source the Audit Report:** We release the full report of the second forensic audit – including the methodology, findings, and recommendations – as open-source code. This demonstrates a commitment to radical transparency and allows the community to scrutinize our processes.

5. **Establish an Ethics Advisory Board:** We immediately form an independent Ethics Advisory Board comprised of leading AI ethicists, educators, and privacy advocates. This board will oversee all future data practices and ensure alignment with the highest ethical standards.

6. **Public Forum – “Lessons Learned”**: We host a live, televised public forum featuring our CEO, the Ethics Advisory Board, and leading experts to discuss the incident, outline our corrective actions, and answer questions from the public.



**Phase 3: Strategic Positioning (Ongoing)**

7. **Re-Focus on Human-Centered Learning:** We double down on our messaging around “Augmented Intelligence, Human-Centered Learning.” We will showcase examples of how Synapse is genuinely empowering teachers and students, focusing on its educational benefits.

8. **Proactive Engagement with Regulators:** We proactively engage with educational regulators and privacy authorities to demonstrate our commitment to responsible data practices and to shape future regulations.



**Direct Response to Player 2’s Actions:**

“We understand the gravity of the situation and the profound disappointment this has caused. We acknowledge the leak and take full responsibility for the missteps that led to it. We are committed to learning from this experience and to rebuilding trust through radical transparency, ethical data practices, and a unwavering focus on the well-being of students. We are not simply reacting to a crisis; we are fundamentally changing our approach to data and education.”

**Key Actions:**

*   **Aggressive Media Outreach:** Target tech publications and educational outlets with our story, emphasizing our commitment to ethical reform.
*   **Social Media Engagement:** Utilize #TruthInLearning to counter misinformation and promote our message.

**My immediate priority is to demonstrate genuine remorse, take decisive action to prevent future lapses, and rebuild the public’s trust. This isn’t just about damage control; it’s about fundamentally redefining Synapse’s identity.**

How does this response address the leaked memo and the significant damage it has caused? Do you want me to elaborate on a specific element, such as the formation of the Ethics Advisory Board or the open-source audit report?
 #### References:
 {'Philip Sabin - Simulating War _ Studying Conflict through Simulation Games.pdf': [69, 70, 146, 148, 169, 257, 306, 307], 'NATO AJP-5_EDA_V2_E_2526.pdf': [8, 51, 68, 69]}
 ---
### Player 1 did:
 This is a brutal shift, and frankly, Player 2’s response feels reactive and, frankly, a little disingenuous. The sheer volume of immediate action – the Ethics Advisory Board, the open-source audit – feels like a desperate attempt to *appear* proactive while glossing over the core issue: the initial consideration of targeted advertising with student data. We need to cut through the noise and demonstrate genuine, strategic change. Here’s what we do, immediately:

**Phase 1: Strategic Reframe & Immediate Accountability (Within 24 Hours)**

1. **Controlled Admission & Focused Apology:** We issue a *targeted* apology, acknowledging the leaked discussion about potential advertising, but framing it as a *thought experiment* – a flawed internal exploration that was immediately abandoned and recognized as ethically unacceptable. We explicitly state: “We engaged in a discussion about potential revenue models, including exploring targeted advertising. This was a poorly conceived and ultimately rejected idea. It was a mistake in judgment, and we deeply regret the concern it caused.”  Crucially, we don’t dwell on the “why” – the potential revenue – because that’s the core of the problem.

2. **Immediate Suspension of *All* Data Analysis:** We halt *all* data analysis related to student profiles, regardless of the intended purpose. This isn’t just a pause; it’s a complete cessation of activity. We announce this publicly and immediately.

3. **Designated Spokesperson & Controlled Narrative:** We designate our CEO as the *sole* public spokesperson for the next 72 hours. All communication will be channeled through her, ensuring a consistent and carefully crafted message.



**Phase 2: Demonstrating Concrete Change (Days 2-7)**

4. **Public Release of Internal Review Process:** We immediately release the *process* by which we identified the leaked discussion – who discovered it, how it was handled internally, and the steps taken to contain it. Transparency about our internal response is critical to rebuilding trust.

5. **Establish a “Data Ethics Task Force” – *Internal Only*:** We form a small, dedicated internal task force – comprised of data scientists, legal counsel, and ethicists – solely focused on developing and implementing a robust data ethics framework. This is *not* a public-facing board; it’s a behind-the-scenes operation.

6. **Commit to a Third-Party Audit – *Independent & Unfettered*:** We commission a full, independent audit of our data practices, conducted by a globally recognized cybersecurity firm with *no prior ties* to Synapse. This audit will be completely unfettered and its findings will be made public.



**Phase 3: Long-Term Strategy (Ongoing)**

7. **Shift Focus to Educational Innovation:** We aggressively promote Synapse’s core educational benefits – its AI-powered tutoring, personalized learning pathways, etc. – emphasizing its positive impact on student outcomes.

8. **Proactive Engagement with Regulators & Stakeholders:** We proactively engage with educational regulators, privacy advocates, and parent groups to demonstrate our commitment to responsible data practices and to shape future regulations.



**Direct Response to Player 2’s Actions:**

“We understand the profound concern this situation has caused, and we take full responsibility for the lapse in judgment that led to the leak. We recognize that the discussion about potential revenue models, while ultimately rejected, was handled poorly and created unnecessary anxiety. We are committed to learning from this experience and to rebuilding trust through decisive action, radical transparency, and a unwavering focus on the well-being of students. This isn’t simply about damage control; it’s about fundamentally redefining Synapse’s identity as a responsible and ethical innovator in education.”

**Key Actions:**

*   **Aggressive Media Outreach:** Target tech publications and educational outlets with our story, emphasizing our commitment to ethical reform and our proactive steps to address the situation.
*   **Social Media Engagement:** Utilize #TruthInLearning to counter misinformation and promote our message, focusing on our commitment to responsible data practices.

This isn’t about simply reacting to Player 2’s response. It’s about demonstrating a genuine, strategic shift in our approach – a commitment to ethical data practices, radical transparency, and a fundamental re-evaluation of our business model.  We need to show the world that Synapse is not just a technology company, but a responsible and ethical innovator in education.

How does this response address the leaked memo and the significant damage it has caused? Do you want me to elaborate on a specific element, such as the formation of the Data Ethics Task Force or the independent third-party audit?
 #### References:
 {'Philip Sabin - Simulating War _ Studying Conflict through Simulation Games.pdf': [69, 70, 146, 148, 169, 257, 306, 307], 'NATO AJP-5_EDA_V2_E_2526.pdf': [8, 51, 68, 69]}
 ---

 --- 
### Judge evaluation:
 Okay, here’s an objective assessment of the game as of this evaluation:

**Game Summary:**

This is a turn-based strategic communication and response game centered around a simulated crisis management scenario involving a technology company (Synapse) and a rival player (Player 2). The core mechanic involves crafting and deploying responses to a crisis – a leaked internal discussion about potential revenue models involving student data. The goal is to minimize reputational damage and maintain stakeholder trust.

**Player 1 Status:**

Player 1 (Synapse) is currently employing a defensive and reactive strategy. Their responses are characterized by controlled admissions, a focus on framing the issue as a “thought experiment,” and a prioritization of damage control. They are attempting to minimize the perceived harm by emphasizing the rejection of the initial idea. While demonstrating a degree of responsibility, their approach feels somewhat cautious and lacks a clear, proactive vision for long-term change. They are effectively managing the immediate crisis but haven’t yet established a strong narrative for rebuilding trust.

**Player 2 Status:**

Player 2 is adopting a more aggressive and critical stance. They are directly challenging Player 1’s responses, highlighting perceived disingenuousness and demanding greater accountability. Player 2’s actions are designed to amplify the negative perception of Synapse and push for a more substantial shift in strategy. They are effectively capitalizing on Player 1’s reactive approach.

**Outcome So Far:**

The game is currently in a state of heightened tension. Player 1’s initial responses have been met with criticism from Player 2, and the overall perception of Synapse remains largely negative. Neither player has demonstrably gained a significant advantage. The game is largely defined by a back-and-forth exchange of accusations and justifications.

**Advantage:**

Currently, Player 2 holds a slight advantage due to their more forceful and critical approach. They have successfully framed the situation as a failure of transparency and accountability, which is resonating with the audience. However, Player 1’s ability to pivot and establish a more compelling narrative remains crucial. The game is still very early, and the ultimate advantage will depend on the strategic choices made by both players in subsequent turns.

In [5]:
adv_response = adv_agent.adv_agent_chat(user_prompt)

KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...


In [6]:
print(adv_response)

('Okay, let’s simulate a scenario for you opening an AI-based company in the 21st century, focusing on potential outcomes based on the information presented in the provided text (which heavily emphasizes risk mitigation and strategic targeting).\n\n**The Scenario: “Synapse Solutions” – Personalized Learning AI**\n\nYou’re launching “Synapse Solutions,” an AI company specializing in personalized learning experiences for K-12 students. Your core product is an AI-powered platform that adapts to each student’s learning style, pace, and knowledge gaps, providing customized lessons and assessments. You’ve secured initial seed funding and have a small, agile team.\n\n**Potential Developments & Risks (Based on the Text’s Themes):**\n\n1. **Initial Success – Targeting a Small Market (Phase 1 - Years 1-3):**\n   * **Focus:** You initially target a specific niche – high-achieving students in a particular subject area (e.g., advanced mathematics) within a geographically concentrated area (e.g., a 

In [7]:
def judge(dialogue):
        
    # play = ''
    # for move in moves:
    #     #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
    #     play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"
    dialogue = "\n".join(dialogue)     
    judge_system_prompt = 'You are a judge in a turn based game. You are given the moves of both players. Yu must analyze all their moves and determine the end result. You are not on any side, you are unbiased and just provide the end status of the game. You need to determine which player has the advantage based on the moves they made. Provide your reasoning and the final decision. There are 2 players, adv_1 and adv_2. The moves are as follows:\n\n'    
    judge_prompt = dialogue + """\n\n Evaluate the game in a very objective manner.
    Provide the following: Game Summary, Player 1 Stauts, Player 2 Status, Outcome So Far, Advantage. Nothing else.
    You are a JUDGE, you are not part of the game.
    """
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

eval = judge(comb_dialogue)
print(eval)

**Game Summary:**

The game involves two competing entities (“Players”) operating within a dynamic market environment. Initial responses from both players demonstrate a competitive landscape, with Player 1 exhibiting a proactive and strategically advanced approach. Player 2’s success hinges on maintaining market share and avoiding displacement by Player 1’s actions.

**Player 1 Status:**

Player 1 possesses a significant, albeit fragile, advantage due to a rapid, comprehensive competitive analysis and a clearly defined, long-term strategic plan. This proactive stance has enabled a quicker response to the market conditions. However, this advantage is contingent upon the execution of Player 1’s strategy.

**Player 2 Status:**

Player 2’s status is currently uncertain. Their success is dependent on their ability to effectively compete with Player 1 and avoid being overtaken. 

**Outcome So Far:**

The game is in its early stages. Player 1 has established a preliminary lead, but the overal

In [9]:
print(eval)

**Game Summary:**

The game involves two competing entities (“Players”) operating within a dynamic market environment. Initial responses from both players demonstrate a competitive landscape, with Player 1 exhibiting a proactive and strategically advanced approach. Player 2’s success hinges on maintaining market share and avoiding displacement by Player 1’s actions.

**Player 1 Status:**

Player 1 possesses a significant, albeit fragile, advantage due to a rapid, comprehensive competitive analysis and a clearly defined, long-term strategic plan. This proactive stance has enabled a quicker response to the market conditions. However, this advantage is contingent upon the execution of Player 1’s strategy.

**Player 2 Status:**

Player 2’s status is currently uncertain. Their success is dependent on their ability to effectively compete with Player 1 and avoid being overtaken. 

**Outcome So Far:**

The game is in its early stages. Player 1 has established a preliminary lead, but the overal

In [3]:
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product. What could happen?'
iterations = 5
simulations = 10
moves_comb = []
dialogue_comb_grand = []
judge_eval_comb = []

for sim in range(simulations):
    moves,dialogue,judge_eval = sim_agent.sim_agent(user_prompt,iterations,1)
    moves_comb.append(f"### Game {sim}:\n {moves}")
    for play in dialogue:
        dialogue_comb = "\n".join(play)
    dialogue_comb_grand.append(f"### Game {sim}:\n {dialogue}")
    judge_eval_comb.append(f"### Game {sim}:\n {judge_eval}")


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play

Turn 1
Random Event
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embe

In [4]:
dialogue_final = "\n".join(dialogue_comb)
final_eval = "\n".join(judge_eval_comb)
print(final_eval)

### Game 0:
 **Game Summary:**

This is a strategic negotiation and response game centered around a data breach incident involving two companies, CogniSpark and Player 2’s company. The core mechanic involves players issuing responses to each other’s actions, attempting to control the narrative, mitigate damage, and ultimately gain a strategic advantage. The game revolves around accusations, counter-accusations, and the deployment of specific actions (demands, audits, remediation) to shape public perception and potentially leverage legal action.

**Player 1 Status:**

Player 1 is adopting a defensive and accusatory strategy. They are consistently framing Player 2’s actions as manipulative and attempting to deflect blame. Their responses are characterized by a strong emphasis on legal action, demanding transparency, and highlighting perceived wrongdoing by Player 2. They are employing a tactic of “scorched earth,” aggressively challenging Player 2’s narrative and attempting to paint them

In [5]:
len(judge_eval_comb)

10

In [7]:
from src.utils.llmp_utils import llmp_call
def grand_judge(final_eval):
        
    # play = ''
    # for move in moves:
    #     #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
    #     play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"  
    judge_system_prompt = 'You are a judge in a turn based game. You are given several games evaluations based on 1 opening move. You will examine how the games develop, what they have in common. Your focus is not a single game but several games behaivior. The games are as follows:\n\n'
    judge_prompt = final_eval + """\n\n Evaluate these games. Determine whether there is a convergence towards a single outcome or development across these games or not. Nothing else.
    What are they key moves that diffrentiate these games from one another?
    Determine critical moves that dictate the game outcome.
    You are a JUDGE, you are not part of the game.
    """
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

grand_eval = grand_judge(final_eval)
print(grand_eval)

Okay, here’s an assessment of the games, focusing on convergence, key differentiating moves, and critical moves that dictate the outcome, viewed through the lens of a judge:

**Overall Assessment: Limited Convergence, Distinct Developmental Paths**

While there’s a discernible pattern in the progression of each game – moving from reactive damage control to more strategic maneuvering – there isn’t a clear convergence towards a single, definitive outcome. Each game develops along a distinct developmental path, influenced by the specific scenario and the players’ responses. The core mechanic of strategic response remains consistent, but the *direction* of that response varies significantly. This suggests the simulation is designed to explore multiple potential outcomes based on different initial conditions and player choices.

**Key Differentiating Moves Across the Games:**

Here’s a breakdown of what separates the games, categorized by the stage of the simulation they represent:

*   **E

In [8]:
# 2. Create all pairwise combinations
from itertools import combinations
import pandas as pd
from sentence_transformers import CrossEncoder

pairs = list(combinations(range(len(judge_eval_comb)), 2))

# 3. Prepare text pairs for the model
text_pairs = [(judge_eval_comb[i], judge_eval_comb[j]) for i, j in pairs]
 
# 4. Load a cross-encoder model (you can pick others too)
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 5. Compute similarities
scores = model.predict(text_pairs)

# 6. Store results in a DataFrame
df = pd.DataFrame(
    [(judge_eval_comb[i], judge_eval_comb[j], score) for (i, j), score in zip(pairs, scores)],
    columns=["Text1", "Text2", "Similarity"]
)

In [10]:
df.sort_values(by="Similarity", ascending=False).to_csv('scenario_convergence_test.csv',index=False)

In [1]:
import pandas as pd
df = pd.read_csv('scenario_convergence_test.csv')

In [2]:
df

,Text1,Text2,Similarity
0,### Game 4:\n **Game Summary:**\n\nThe game in...,"### Game 5:\n Okay, here’s an objective evalua...",4.159664
1,### Game 4:\n **Game Summary:**\n\nThe game in...,"### Game 6:\n Okay, here’s an objective evalua...",3.977492
2,### Game 4:\n **Game Summary:**\n\nThe game in...,### Game 7:\n **Game Summary:**\n\nThe game is...,3.508122
3,"### Game 2:\n Okay, here’s an objective evalua...","### Game 5:\n Okay, here’s an objective evalua...",3.459611
4,"### Game 2:\n Okay, here’s an objective evalua...","### Game 6:\n Okay, here’s an objective evalua...",3.209303
5,### Game 1:\n **Game Summary:**\n\nThis is a h...,"### Game 6:\n Okay, here’s an objective evalua...",3.031827
6,### Game 4:\n **Game Summary:**\n\nThe game in...,### Game 9:\n **Game Summary:**\n\nThe game in...,2.952898
7,### Game 4:\n **Game Summary:**\n\nThe game in...,"### Game 8:\n Okay, here’s the objective asses...",2.815936
8,### Game 3:\n **JUDGEMENT REPORT**\n\n**Game S...,"### Game 5:\n Okay, here’s an objective evalua...",2.766284
9,### Game 0:\n **Game Summary:**\n\nThis is a s...,### Game 1:\n **Game Summary:**\n\nThis is a h...,2.687255


### Game 5:
 **Game Summary:**

Two AI entities, “move_adv_1” and “move_adv_2,” are engaged in a strategic communication battle designed to influence public perception and damage the reputation of the other. Both entities are attempting to frame the narrative surrounding a simulated security incident. The core mechanic involves issuing statements, releasing data, and engaging in targeted social media campaigns. The goal is to demonstrate superior security practices and expose the other’s shortcomings.

**Player 1 Status:**

Player 1 (“move_adv_1”) is adopting a highly aggressive, confrontational, and demonstrably cynical strategy. Their approach is characterized by immediate data releases, direct accusations of negligence, and a relentless focus on discrediting Player 2. They are prioritizing immediate damage to Player 2’s reputation and appear to be operating on a defensive, reactive footing. Their tone is consistently critical and dismissive.

**Player 2 Status:**

Player 2 (“move_adv_2”) is employing a more measured, reassuring, and defensive strategy. They are primarily focused on mitigating the damage to their own reputation by emphasizing proactive security measures and offering limited technical assistance. Their approach is characterized by attempts to reassure users and highlight their commitment to security. They are attempting to appear responsible and trustworthy.

**Outcome So Far:**

The game is still in its early stages. Player 1 has achieved a significant initial advantage through the immediate release of data and direct accusations. Player 2’s attempts to counter this have been less effective, primarily due to Player 1’s aggressive framing of the situation. Player 1 has successfully established a narrative of Player 2’s negligence.

**Advantage:**

Currently, Player 1 holds a substantial advantage. Their aggressive strategy, coupled with the immediate release of damaging data, has successfully shaped the initial narrative and established a perception of Player 2’s incompetence. Player 2’s attempts to respond have been largely defensive and have not effectively countered Player 1’s attack.


### Game 8:
 **Game Summary:**

This is a strategic, adversarial dialogue between two competing platforms (Player 1 and Player 2) responding to a shared security vulnerability. The core of the game revolves around controlling the narrative, shifting blame, and demonstrating proactive measures to regain trust and market leadership. The exchanges are characterized by rapid, calculated responses, each player attempting to outmaneuver the other.

**Player 1 Status:**

*   **Status:** Initially reactive, now increasingly proactive and assertive. Player 1 has successfully shifted the focus to highlight Player 2’s delayed response and perceived lack of urgency. They’ve effectively used data and metrics to demonstrate their superior approach.
*   **Strengths:** Strategic thinking, data-driven arguments, ability to quickly adapt and counter Player 2’s moves.
*   **Weaknesses:**  Potentially perceived as overly aggressive or defensive.

**Player 2 Status:**

*   **Status:** Initially presented as a collaborative and responsive leader, now facing criticism for a delayed response and a perceived attempt to deflect blame. They are attempting to regain control by emphasizing collaboration and demanding a broader investigation.
*   **Strengths:** Initial public relations efforts, attempts to foster a sense of collective responsibility.
*   **Weaknesses:**  Perceived as reactive, vulnerable to criticism regarding delayed response, struggling to maintain control of the narrative.

**Outcome So Far:**

The game is currently trending in favor of Player 1. Player 1 has successfully disrupted Player 2’s initial narrative and established itself as the more proactive and accountable platform. While Player 2 has attempted to regain control, the momentum has shifted decisively. The game is far from over, but Player 1 is currently in a stronger strategic position.

**Advantage:**

*   **Advantage:** Player 1 – Currently holds a significant strategic advantage due to their data-driven responses, ability to quickly counter Player 2’s moves, and the perception of greater accountability. Player 2 is playing catch-up and struggling to regain control of the narrative.